# Project : Review Intelligence Engine

By Akshat Kesharwani | Free tools only (Google Colab + free-tier APIs)

## Problem Statement

Companies get way too many reviews to read by hand, but there are real patterns buried in there. Teams at places like American Express use classifiers to automatically flag negative feedback and route it to the right place. Here we fine-tune RoBERTa for sentiment, then add a simple keyword-based layer that buckets negative reviews into categories like shipping, quality, or pricing.

### SECTION 1: Install & Import Libraries

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn openai

import os
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ---- Small AI helper: call Groq (free) ----
# Only using Groq here -- it's free, fast, and just needs an API key,
# no extra setup like Ollama needs.
# Get a free key at https://console.groq.com/keys
# (Update: as of June 2026 Groq retired the old Llama-3.3-70B model,
# the current one to use is "openai/gpt-oss-120b".)
from getpass import getpass
from openai import OpenAI

groq_key = getpass("Enter your Groq API key: ")
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_key)

def call_llm(prompt, max_tokens=300):
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.5 MB/s eta 0:00:00
Enter your Groq API key: ··········


### SECTION 2: Load Review Dataset (with robust fallback)

In [2]:
# heads up: the plain "amazon_polarity" repo uses the old loading-script
# way of doing things, which newer `datasets` versions block. HuggingFace's
# proper parquet version is "fancyzhx/amazon_polarity" -- we try that first, then the old
# name, then fall back to small built-in synthetic reviews.
def robust_load_dataset(repo_candidates, split="train"):
    for repo_id in repo_candidates:
        try:
            print(f"[dataset] Trying load_dataset('{repo_id}', split='{split}')...")
            ds = load_dataset(repo_id, split=split)
            print(f"[dataset] Loaded '{repo_id}' successfully ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] '{repo_id}' standard load failed: {type(e).__name__}: {e}")
        try:
            parquet_path = f"hf://datasets/{repo_id}@refs/convert/parquet/default/{split}/*.parquet"
            print(f"[dataset] Trying Parquet fallback: {parquet_path}")
            ds = load_dataset("parquet", data_files=parquet_path, split="train")
            print(f"[dataset] Parquet fallback for '{repo_id}' succeeded ({len(ds)} rows).")
            return ds
        except Exception as e:
            print(f"[dataset] Parquet fallback for '{repo_id}' failed: {type(e).__name__}: {e}")
    return None

raw = robust_load_dataset(["fancyzhx/amazon_polarity", "amazon_polarity"], split="train")

if raw is not None:
    raw = raw.shuffle(seed=42).select(range(min(100000, len(raw))))
    df = pd.DataFrame(raw)[["content", "label"]].rename(columns={"content": "text"})
else:
    # ---- Synthetic offline fallback -- guarantees the notebook always runs ----
    SYNTHETIC_REVIEWS = [
        {"text": "This product broke after two days of light use. Total waste of money, arrived late too.", "label": 0},
        {"text": "Exactly what I needed, works perfectly and shipped a day early. Highly recommend!", "label": 1},
        {"text": "Customer service was rude when I asked about a refund. Never buying from this brand again.", "label": 0},
        {"text": "Great value for the price, quality feels much better than expected.", "label": 1},
        {"text": "The item arrived damaged and packaging was flimsy. Very disappointed.", "label": 0},
        {"text": "Fast shipping, well packaged, and the product matches the description perfectly.", "label": 1},
        {"text": "Overpriced for what you get -- I've seen the same thing cheaper elsewhere.", "label": 0},
        {"text": "Been using it daily for a month now and it still works like new. Excellent build quality.", "label": 1},
        {"text": "Support never responded to my emails about a defective unit. Frustrating experience.", "label": 0},
        {"text": "Easy to set up and does exactly what it promises. Would buy again.", "label": 1},
    ]
    expanded = SYNTHETIC_REVIEWS * 30  # replicate so the Trainer has a workable number of steps
    df = pd.DataFrame(expanded)
    print("[dataset] Using synthetic fallback: swap in a real Kaggle/HF dataset before reporting final metrics.")

print("Training reviews:", len(df))
print(df["label"].value_counts())

dataset = Dataset.from_pandas(df).train_test_split(test_size=0.05, seed=42)

[dataset] Trying load_dataset('fancyzhx/amazon_polarity', split='train')...


README.md:   0%|          | 0.00/6.81k [00:00<?, ?B/s]

amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  260MB            

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:           |  0.00B            

amazon_polarity/train-00001-of-00004.par(…): reconstructing file:   0%|          |  0.00B /  258MB            

...

amazon_polarity/test-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  117MB            

amazon_polarity/test-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

[dataset] Loaded 'fancyzhx/amazon_polarity' successfully (3600000 rows).
Training reviews: 100000
label
0    50231
1    49769
Name: count, dtype: int64


### SECTION 3: Tokenize

In [3]:
model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text", "__index_level_0__"] if "__index_level_0__" in dataset["train"].column_names else ["text"])
data_collator = DataCollatorWithPadding(tokenizer)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/95000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

### SECTION 4: Load Pretrained RoBERTa

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### SECTION 5: Fine-Tune

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

use_fp16 = torch.cuda.is_available()  # fp16 only works with a real GPU -- Runtime > Change runtime type > T4 GPU
print("GPU available:", use_fp16)

training_args = TrainingArguments(
    output_dir="./review-intelligence-checkpoints",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,      # kept small so free Colab doesn't run out of RAM      # kept small so free Colab doesn't run out of RAM
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    gradient_accumulation_steps=4,   # makes up for the smaller batch size above
    gradient_checkpointing=True,     # uses less memory, trains a little slower
    save_strategy="epoch",
    save_total_limit=1,
    fp16=use_fp16,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
    data_collator=data_collator, processing_class=tokenizer, compute_metrics=compute_metrics  # newer transformers renamed "tokenizer" to "processing_class"
)
trainer.train()
eval_results = trainer.evaluate()
print("Eval results:", eval_results)

GPU available: True


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.786373,0.180362,0.953800,0.953568
2,0.598266,0.225525,0.956400,0.955977


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.598266,0.225525,2,0.956400,0.955977


Eval results: {'eval_loss': 0.22552505135536194, 'eval_accuracy': 0.9564, 'eval_f1': 0.9559773828756059}


### SECTION 6: Automated Complaint Analysis

In [6]:
# Rule + model hybrid: flag negative reviews, then bucket them by
# complaint category using keyword matching (fast, free, explainable).
COMPLAINT_CATEGORIES = {
    "shipping": ["late", "delayed", "shipping", "delivery", "arrived broken"],
    "quality": ["broke", "defective", "poor quality", "cheap", "flimsy"],
    "customer_service": ["rude", "unhelpful", "no response", "support"],
    "pricing": ["overpriced", "expensive", "not worth", "refund"]
}

def categorize_complaint(text):
    text_lower = text.lower()
    matched = [cat for cat, kws in COMPLAINT_CATEGORIES.items() if any(kw in text_lower for kw in kws)]
    return matched if matched else ["uncategorized"]

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        logits = model(**{k: v.to(model.device) for k, v in inputs.items()}).logits
    pred = torch.argmax(logits, dim=-1).item()
    return "positive" if pred == 1 else "negative"

sample_reviews = df.sample(20, random_state=1)
complaint_summary = {}
for text in sample_reviews["text"]:
    sentiment = predict_sentiment(text)
    if sentiment == "negative":
        for cat in categorize_complaint(text):
            complaint_summary[cat] = complaint_summary.get(cat, 0) + 1

print("Complaint category breakdown (sample):", complaint_summary)

Complaint category breakdown (sample): {'uncategorized': 6, 'quality': 1, 'shipping': 1, 'customer_service': 1, 'pricing': 1}


### SECTION 7: Save Model

In [7]:
model.save_pretrained("./review-intelligence-model")
tokenizer.save_pretrained("./review-intelligence-model")
print("Model saved to ./review-intelligence-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./review-intelligence-model
